# 2段階でランダムフォレストをやっているノート

# Import Library

In [ ]:
#Initial setup
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display

import pandas as pd
import os
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from decimal import Decimal, ROUND_HALF_UP
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LassoCV
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    RocCurveDisplay,
    confusion_matrix,
    accuracy_score,
    precision_score,
    balanced_accuracy_score,
    recall_score,
    f1_score,
    classification_report
)


from sklearn.model_selection import KFold
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.linear_model import LogisticRegression
import urllib.request
import joblib
import json

# Import dataset

In [ ]:
fileDir = "./Data/"
fileName = "pro_data_common_val_Assay.csv"

path = os.path.join(fileDir, fileName)
temp = pd.read_csv(path)
temp.set_index(temp.columns[0],inplace=True)
temp.index.name = None
pro_data = temp
display(pro_data)

In [ ]:
fileDir = "./Data/"
fileName = "pro_data_common_dis_Assay.csv"

path = os.path.join(fileDir, fileName)
temp = pd.read_csv(path)
temp.set_index(temp.columns[0],inplace=True)
temp.index.name = None
pro_data_dis = temp
display(pro_data_dis)

In [ ]:
fileDir = "./Data/"
fileName = "pro_sample_val.csv"

path = os.path.join(fileDir, fileName)
temp = pd.read_csv(path)
temp.set_index(temp.columns[0],inplace=True)
temp.index.name = None
pro_sample = temp
display(pro_sample)

In [ ]:
fileDir = "./Data/"
fileName = "map_OID_Uni_Sym.csv"

path = os.path.join(fileDir, fileName)
IDdata = pd.read_csv(path)
IDdata.set_index(IDdata.columns[0],inplace=True)
IDdata.index.name = None
display(IDdata)

In [ ]:
fileDir = "./Data/"
fileName = "DEP_list.csv"

path = os.path.join(fileDir, fileName)
temp = pd.read_csv(path)
temp.set_index(temp.columns[0],inplace=True)
temp.index.name = None
display(temp)
DEPlist = temp

In [ ]:
fileDir = "./Data/"
fileName = "pro_data_common_dis.csv"

path = os.path.join(fileDir, fileName)
temp = pd.read_csv(path)
temp.set_index(temp.columns[0],inplace=True)
temp.index.name = None
pro_data_dis= temp
display(pro_data_dis)

In [ ]:
fileDir = "../Proteomics/Data/"
fileName = "CA199.csv"

path = os.path.join(fileDir, fileName)
temp = pd.read_csv(path)
temp.set_index(temp.columns[0],inplace=True)
temp.index.name = None
temp = temp.sort_values('SampleID')
temp = temp.set_index('SampleID')
ca199_dis= temp
display(ca199_dis)

In [ ]:
fileDir = "./Data/"
fileName = "pro_sample_dis.csv"

path = os.path.join(fileDir, fileName)
temp = pd.read_csv(path)
temp.set_index(temp.columns[0],inplace=True)
temp.index.name = None
pro_sample_dis = temp
display(pro_sample_dis)

# モデル学習

log変換、欠損値の処理、標準化済み

## ①全プロテオミクスver

In [ ]:
X = pro_data
y = pro_sample["CaCo_Status"]

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
import time
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
aucs = []
fig, ax = plt.subplots()

# 各foldの予測結果を保存
y_prob_all = np.full(len(y), np.nan)
y_true_all = np.full(len(y), np.nan)
feature_importances = []

# 追加：各foldの評価指標・最適パラメータを保存
fold_metrics = []
best_params_all = []

save_dir = "./RF_Data/RF_status_all"
os.makedirs(save_dir, exist_ok=True)

for i, (train_index, test_index) in enumerate(skf.split(X, y)):

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    param_grid = {
        'n_estimators': [50, 100, 300, 500],
        "max_depth": [None, 3, 5, 10, 15],
        'min_samples_leaf': [1, 2, 4],
        'max_features': [None, 'sqrt'],
        'class_weight': ['balanced']
    }

    model = RandomForestClassifier(
        criterion='gini',
        random_state=123,
        n_jobs=1
    )

    cv_gen = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

    modelcv = GridSearchCV(
        model,
        param_grid=param_grid,
        scoring='roc_auc',
        n_jobs=-1,
        refit=True,
        cv=cv_gen,
        verbose=0,
        pre_dispatch='2*n_jobs',
        error_score=np.nan,
        return_train_score=False
    )

    start = time.time()
    modelcv.fit(X_train, y_train)
    elapsed = time.time() - start

    print(f"1回のGridSearchCV: {elapsed / 60:.2f} 分")
    print(f"全5 foldの推定時間: {elapsed * 5 / 60:.2f} 分")
    print('Best score: ', modelcv.best_score_)
    print('Best hyperparameters: ', modelcv.best_params_)

    # 追加：各foldの最適パラメータを保存
    best_params_all.append({
        "Fold": i + 1,
        "Best_Params": modelcv.best_params_
    })

    dataframe = pd.DataFrame(
        modelcv.cv_results_
    ).sort_values('mean_test_score', ascending=False)
    display(dataframe)

    model = modelcv.best_estimator_

    # 予測確率とAUC
    y_prob = model.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, y_prob)
    aucs.append(score)

    # OOF予測を保存
    y_prob_all[test_index] = y_prob
    y_true_all[test_index] = np.asarray(y_test)

    # 各foldの特徴量重要度
    feature_importances.append(model.feature_importances_)

    # 追加：各foldの評価指標を計算
    y_pred = (y_prob >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test, y_pred, labels=[0, 1]
    ).ravel()

    fold_metrics.append({
        "Fold": i + 1,
        "AUC": score,
        "Accuracy": (tp + tn) / (tp + tn + fp + fn),
        "Sensitivity": tp / (tp + fn) if tp + fn else np.nan,
        "Specificity": tn / (tn + fp) if tn + fp else np.nan,
        "False Positive Rate": fp / (fp + tn) if fp + tn else np.nan,
        "False Negative Rate": fn / (fn + tp) if fn + tp else np.nan,
        "PPV": tp / (tp + fp) if tp + fp else np.nan,
        "NPV": tn / (tn + fn) if tn + fn else np.nan
    })

    print('AUC: ', score)

    RocCurveDisplay.from_predictions(
        y_test,
        y_prob,
        name=f"Fold {i+1}",
        ax=ax
    )

print(f"5-Fold平均AUC: {np.mean(aucs):.4f} ± {np.std(aucs, ddof=1):.4f}")
plt.show()

# ==========================================
# 全foldの予測結果を保存
# ==========================================
y_true_all = y_true_all.astype(int)
y_pred_all = (y_prob_all >= 0.5).astype(int)

oof_df = pd.DataFrame({
    "TrueClass": y_true_all,
    "PredictedProbability": y_prob_all,
    "PredictedClass": y_pred_all
}, index=X.index)

oof_df.index.name = "SampleID"
oof_df.to_csv(f"{save_dir}/oof_predictions.csv")

# ==========================================
# 各foldの評価指標と平均・標準偏差を保存
# ==========================================
fold_metrics_df = pd.DataFrame(fold_metrics)

# 各foldの結果
display(fold_metrics_df.round(4))
fold_metrics_df.to_csv(
    f"{save_dir}/metrics_each_fold.csv",
    index=False
)

# 5-foldの平均と標準偏差
metrics_df = pd.DataFrame({
    "Metric": fold_metrics_df.columns.drop("Fold"),
    "Mean": [
        fold_metrics_df[col].mean()
        for col in fold_metrics_df.columns.drop("Fold")
    ],
    "SD": [
        fold_metrics_df[col].std(ddof=1)
        for col in fold_metrics_df.columns.drop("Fold")
    ]
})

display(metrics_df.round(4))
metrics_df.to_csv(f"{save_dir}/metrics.csv", index=False)

# ==========================================
# foldごとの特徴量重要度の平均を保存
# ==========================================
importance_df = pd.DataFrame({
    "Protein": X.columns,
    "Importance": np.mean(feature_importances, axis=0),
    "SD_Importance": np.std(feature_importances, axis=0, ddof=1)
}).sort_values("Importance", ascending=False)

importance_df.to_csv(
    f"{save_dir}/feature_importance.csv",
    index=False
)

display(importance_df.head(20))

# ==========================================
# 各foldの最適パラメータ5つを保存
# ==========================================
with open(f"{save_dir}/best_parameters.json", "w") as f:
    json.dump(
        best_params_all,
        f,
        indent=2,
        default=str
    )

print("結果を保存しました:", save_dir)

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

save_dir = "./RF_Data/RF_status_all"

# ==========================================
# 1. 各foldの評価指標
# ==========================================
fold_metrics_df = pd.read_csv(
    f"{save_dir}/metrics_each_fold.csv"
)

display(fold_metrics_df.round(4))

# ==========================================
# 2. 5-foldの平均・標準偏差
# ==========================================
metrics_df = pd.read_csv(
    f"{save_dir}/metrics.csv"
)

display(metrics_df.round(4))

# ==========================================
# 3. ROC曲線（各foldのOOF予測から作成）
# ==========================================
oof_df = pd.read_csv(
    f"{save_dir}/oof_predictions.csv",
    index_col="SampleID"
)

y_true = oof_df["TrueClass"].to_numpy()
y_prob = oof_df["PredictedProbability"].to_numpy()
y_pred = oof_df["PredictedClass"].to_numpy()

fpr, tpr, thresholds = roc_curve(y_true, y_prob)
auc = roc_auc_score(y_true, y_prob)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f"OOF ROC (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

# ==========================================
# 4. 混同行列
# ==========================================
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[0, 1]
)
disp.plot()
plt.title("Confusion Matrix (OOF)")
plt.show()

# ==========================================
# 5. 偽陽性・偽陰性のサンプルを確認
# ==========================================
fp_df = oof_df[
    (oof_df["TrueClass"] == 0) &
    (oof_df["PredictedClass"] == 1)
]

fn_df = oof_df[
    (oof_df["TrueClass"] == 1) &
    (oof_df["PredictedClass"] == 0)
]

print("偽陽性（False Positive）:", len(fp_df), "samples")
display(fp_df)

print("偽陰性（False Negative）:", len(fn_df), "samples")
display(fn_df)

# 必要ならCSVとして個別に保存
fp_df.to_csv(f"{save_dir}/false_positive_samples.csv")
fn_df.to_csv(f"{save_dir}/false_negative_samples.csv")

# ==========================================
# 6. 特徴量重要度
# ==========================================
importance_df = pd.read_csv(
    f"{save_dir}/feature_importance.csv"
)

display(importance_df.head(20))

top_n = 20
top_features = importance_df.head(top_n).sort_values(
    "Importance", ascending=True
)

plt.figure(figsize=(8, 7))
plt.barh(
    top_features["Protein"],
    top_features["Importance"],
    xerr=top_features["SD_Importance"]
)
plt.xlabel("Mean Feature Importance")
plt.ylabel("Protein")
plt.title(f"Top {top_n} Feature Importances")
plt.tight_layout()
plt.show()

# ==========================================
# 7. 各foldの最適パラメータ
# ==========================================
with open(
    f"{save_dir}/best_parameters.json",
    "r"
) as f:
    best_params_all = json.load(f)

best_params_df = pd.DataFrame([
    {
        "Fold": item["Fold"],
        **item["Best_Params"]
    }
    for item in best_params_all
])

display(best_params_df)

In [ ]:

# ==========================================
# 1. データ準備
# ==========================================

X = pro_data.loc[:, pro_data.columns.isin(DEPlist.index)]

meta = pro_sample.loc[sample_ids].copy()

# progress_groupを文字列に統一
pg = meta["progress_group"].astype(str).str.strip().str.lower()

stage_label = pd.Series(index=meta.index, dtype="object")

# Control
stage_label.loc[meta["CaCo_Status"] == 0] = "Control"
stage_label.loc[pg.isin(["0"])] = "Control"
# Early
stage_label.loc[pg.isin(["1"])] = "Early"

# Advanced
stage_label.loc[pg.isin(["2"])] = "Advanced"


y_stage = stage_label.map({
    "Control": 0,
    "Early": 1,
    "Advanced": 2
}).astype(int)

print("全サンプル数:", len(X))
print(stage_label.value_counts())

# ==========================================
# 2. 第1段階の保存結果を読み込む（再学習なし）
# ==========================================
stage1_dir = "./RF_Data/RF_status_all"

oof_stage1 = pd.read_csv(
    f"{stage1_dir}/oof_predictions.csv",
    index_col="SampleID"
)

metrics_stage1 = pd.read_csv(
    f"{stage1_dir}/metrics.csv"
)

print("第1段階の評価指標:")
display(metrics_stage1)

# ==========================================
# 3. 第2段階：Early vs Advanced
# ==========================================
predicted_cancer_ids = oof_stage1.index[
    oof_stage1["PredictedClass"] == 1
]

# 実際の進行度がEarlyまたはAdvancedであることも条件にする
stage2_mask = (
    X.index.isin(predicted_cancer_ids)
    & y_stage.isin([1, 2])
)

X_stage2 = X.loc[stage2_mask].copy()
y_stage2 = y_stage.loc[stage2_mask].copy()

print("第2段階のサンプル数（0と予測されたものを除いている）:", len(X_stage2))
print("Early / Advanced:")
print(y_stage2.value_counts().sort_index())

# 保存先
save_dir = "./RF_Data/RF_progress_all"
os.makedirs(save_dir, exist_ok=True)


# ==========================================
# 4. 交差検証の設定
# ==========================================
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)

param_grid = {
    "n_estimators": [100, 300, 500],
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_leaf": [2, 4, 6, 8],
    "max_features": ["sqrt", 0.3, 0.5, None],
    "class_weight": ["balanced"]
}

fold_metrics = []
best_params_all = []
feature_importances = []

y_prob_all = np.full(len(y_stage2), np.nan)
y_true_all = np.full(len(y_stage2), np.nan)

fig, ax = plt.subplots(figsize=(7, 7))

# ==========================================
# 5. 5-fold交差検証
# ==========================================
for i, (train_index, test_index) in enumerate(
    skf.split(X_stage2, y_stage2)
):

    X_train = X_stage2.iloc[train_index]
    X_test = X_stage2.iloc[test_index]

    y_train = y_stage2.iloc[train_index]
    y_test = y_stage2.iloc[test_index]

    model = RandomForestClassifier(
        criterion="gini",
        random_state=123,
        n_jobs=1
    )

    cv_gen = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=123
    )

    modelcv = GridSearchCV(
        model,
        param_grid=param_grid,
        scoring="balanced_accuracy",
        n_jobs=-1,
        refit=True,
        cv=cv_gen,
        verbose=0,
        pre_dispatch="2*n_jobs",
        error_score=np.nan,
        return_train_score=False
    )

    start = time.time()
    modelcv.fit(X_train, y_train)
    elapsed = time.time() - start

    print(f"\nFold {i+1}")
    print(f"GridSearchCV時間: {elapsed / 60:.2f} 分")
    print("Best score:", modelcv.best_score_)
    print("Best parameters:", modelcv.best_params_)

    # 各foldの最適パラメータを保存
    best_params_all.append({
        "Fold": i + 1,
        "Best_Params": modelcv.best_params_
    })

    # 必要なら各foldのGridSearch結果も保存
    cv_results_df = pd.DataFrame(
        modelcv.cv_results_
    ).sort_values("mean_test_score", ascending=False)

    cv_results_df.to_csv(
        f"{save_dir}/gridsearch_fold{i+1}.csv",
        index=False
    )

    model = modelcv.best_estimator_

    # 予測確率：Advanced（ラベル2）である確率
    advanced_col = list(model.classes_).index(2)
    y_prob = model.predict_proba(X_test)[:, advanced_col]

    # 予測クラス
    y_pred = model.predict(X_test)

    # OOF予測を保存
    y_prob_all[test_index] = y_prob
    y_true_all[test_index] = y_test.to_numpy()

    # 特徴量重要度
    feature_importances.append(model.feature_importances_)

    # AUC（Advancedを陽性クラスとする）
    score = roc_auc_score(
        (y_test == 2).astype(int),
        y_prob
    )

    # 混同行列：Early=1、Advanced=2
    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred,
        labels=[1, 2]
    ).ravel()

    fold_metrics.append({
        "Fold": i + 1,
        "AUC": score,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
        "Macro F1": f1_score(
            y_test, y_pred,
            labels=[1, 2],
            average="macro",
            zero_division=0
        ),
        "Sensitivity": tp / (tp + fn) if tp + fn else np.nan,
        "Specificity": tn / (tn + fp) if tn + fp else np.nan,
        "False Positive Rate": fp / (fp + tn) if fp + tn else np.nan,
        "False Negative Rate": fn / (fn + tp) if fn + tp else np.nan,
        "PPV": tp / (tp + fp) if tp + fp else np.nan,
        "NPV": tn / (tn + fn) if tn + fn else np.nan
    })

    print("AUC:", score)

    RocCurveDisplay.from_predictions(
        (y_test == 2).astype(int),
        y_prob,
        name=f"Fold {i+1}",
        ax=ax
    )

print("\n5-fold平均AUC:",
      f"{np.mean([m['AUC'] for m in fold_metrics]):.4f} ± "
      f"{np.std([m['AUC'] for m in fold_metrics], ddof=1):.4f}")

plt.title("Early vs Advanced ROC")
plt.show()

# ==========================================
# 6. OOF予測を保存
# ==========================================
y_true_all = y_true_all.astype(int)
y_pred_all = np.where(y_prob_all >= 0.5, 2, 1)

oof_df = pd.DataFrame({
    "TrueClass": y_true_all,
    "PredictedProbability_Advanced": y_prob_all,
    "PredictedClass": y_pred_all
}, index=X_stage2.index)

oof_df.index.name = "SampleID"

oof_df.to_csv(
    f"{save_dir}/oof_predictions.csv"
)

# ==========================================
# 7. 各foldの評価指標を保存
# ==========================================
fold_metrics_df = pd.DataFrame(fold_metrics)

display(fold_metrics_df.round(4))

fold_metrics_df.to_csv(
    f"{save_dir}/metrics_each_fold.csv",
    index=False
)

# 平均・標準偏差
metric_cols = fold_metrics_df.columns.drop("Fold")

metrics_df = pd.DataFrame({
    "Metric": metric_cols,
    "Mean": [
        fold_metrics_df[col].mean()
        for col in metric_cols
    ],
    "SD": [
        fold_metrics_df[col].std(ddof=1)
        for col in metric_cols
    ]
})

display(metrics_df.round(4))

metrics_df.to_csv(
    f"{save_dir}/metrics.csv",
    index=False
)

# ==========================================
# 8. 特徴量重要度を保存
# ==========================================
importance_df = pd.DataFrame({
    "Protein": X_stage2.columns,
    "Importance": np.mean(feature_importances, axis=0),
    "SD_Importance": np.std(
        feature_importances,
        axis=0,
        ddof=1
    )
}).sort_values("Importance", ascending=False)

importance_df.to_csv(
    f"{save_dir}/feature_importance.csv",
    index=False
)

display(importance_df.head(20))

# ==========================================
# 9. 各foldの最適パラメータを保存
# ==========================================
with open(
    f"{save_dir}/best_parameters.json",
    "w"
) as f:
    json.dump(
        best_params_all,
        f,
        indent=2,
        default=str
    )

print("結果を保存しました:", save_dir)

## ②DEP listのもののみver

In [ ]:
display(DEPlist)

In [ ]:
X = pro_data.loc[:, pro_data.columns.isin(DEPlist.index)]
y = pro_sample["CaCo_Status"]

print(DEPlist.shape)
print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
import time
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
aucs = []
fig, ax = plt.subplots()

# 各foldの予測結果を保存
y_prob_all = np.full(len(y), np.nan)
y_true_all = np.full(len(y), np.nan)
feature_importances = []

# 追加：各foldの評価指標・最適パラメータを保存
fold_metrics = []
best_params_all = []

save_dir = "./RF_Data/RF_status_DEP"
os.makedirs(save_dir, exist_ok=True)

for i, (train_index, test_index) in enumerate(skf.split(X, y)):

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    param_grid = {
        'n_estimators': [50, 100, 300, 500],
        "max_depth": [None, 3, 5, 10, 15],
        'min_samples_leaf': [1, 2, 4],
        'max_features': [None, 'sqrt'],
        'class_weight': ['balanced']
    }


    model = RandomForestClassifier(
        criterion='gini',
        random_state=123,
        n_jobs=1
    )

    cv_gen = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

    modelcv = GridSearchCV(
        model,
        param_grid=param_grid,
        scoring='roc_auc',
        n_jobs=-1,
        refit=True,
        cv=cv_gen,
        verbose=0,
        pre_dispatch='2*n_jobs',
        error_score=np.nan,
        return_train_score=False
    )

    start = time.time()
    modelcv.fit(X_train, y_train)
    elapsed = time.time() - start

    print(f"1回のGridSearchCV: {elapsed / 60:.2f} 分")
    print(f"全5 foldの推定時間: {elapsed * 5 / 60:.2f} 分")
    print('Best score: ', modelcv.best_score_)
    print('Best hyperparameters: ', modelcv.best_params_)

    # 追加：各foldの最適パラメータを保存
    best_params_all.append({
        "Fold": i + 1,
        "Best_Params": modelcv.best_params_
    })

    dataframe = pd.DataFrame(
        modelcv.cv_results_
    ).sort_values('mean_test_score', ascending=False)
    display(dataframe)

    model = modelcv.best_estimator_

    # 予測確率とAUC
    y_prob = model.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, y_prob)
    aucs.append(score)

    # OOF予測を保存
    y_prob_all[test_index] = y_prob
    y_true_all[test_index] = np.asarray(y_test)

    # 各foldの特徴量重要度
    feature_importances.append(model.feature_importances_)

    # 追加：各foldの評価指標を計算
    y_pred = (y_prob >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test, y_pred, labels=[0, 1]
    ).ravel()

    fold_metrics.append({
        "Fold": i + 1,
        "AUC": score,
        "Accuracy": (tp + tn) / (tp + tn + fp + fn),
        "Sensitivity": tp / (tp + fn) if tp + fn else np.nan,
        "Specificity": tn / (tn + fp) if tn + fp else np.nan,
        "False Positive Rate": fp / (fp + tn) if fp + tn else np.nan,
        "False Negative Rate": fn / (fn + tp) if fn + tp else np.nan,
        "PPV": tp / (tp + fp) if tp + fp else np.nan,
        "NPV": tn / (tn + fn) if tn + fn else np.nan
    })

    print('AUC: ', score)

    RocCurveDisplay.from_predictions(
        y_test,
        y_prob,
        name=f"Fold {i+1}",
        ax=ax
    )

print(f"5-Fold平均AUC: {np.mean(aucs):.4f} ± {np.std(aucs, ddof=1):.4f}")
plt.show()

# ==========================================
# 全foldの予測結果を保存
# ==========================================
y_true_all = y_true_all.astype(int)
y_pred_all = (y_prob_all >= 0.5).astype(int)

oof_df = pd.DataFrame({
    "TrueClass": y_true_all,
    "PredictedProbability": y_prob_all,
    "PredictedClass": y_pred_all
}, index=X.index)

oof_df.index.name = "SampleID"
oof_df.to_csv(f"{save_dir}/oof_predictions.csv")

# ==========================================
# 各foldの評価指標と平均・標準偏差を保存
# ==========================================
fold_metrics_df = pd.DataFrame(fold_metrics)

# 各foldの結果
display(fold_metrics_df.round(4))
fold_metrics_df.to_csv(
    f"{save_dir}/metrics_each_fold.csv",
    index=False
)

# 5-foldの平均と標準偏差
metrics_df = pd.DataFrame({
    "Metric": fold_metrics_df.columns.drop("Fold"),
    "Mean": [
        fold_metrics_df[col].mean()
        for col in fold_metrics_df.columns.drop("Fold")
    ],
    "SD": [
        fold_metrics_df[col].std(ddof=1)
        for col in fold_metrics_df.columns.drop("Fold")
    ]
})

display(metrics_df.round(4))
metrics_df.to_csv(f"{save_dir}/metrics.csv", index=False)

# ==========================================
# foldごとの特徴量重要度の平均を保存
# ==========================================
importance_df = pd.DataFrame({
    "Protein": X.columns,
    "Importance": np.mean(feature_importances, axis=0),
    "SD_Importance": np.std(feature_importances, axis=0, ddof=1)
}).sort_values("Importance", ascending=False)

importance_df.to_csv(
    f"{save_dir}/feature_importance.csv",
    index=False
)

display(importance_df.head(20))

# ==========================================
# 各foldの最適パラメータ5つを保存
# ==========================================
with open(f"{save_dir}/best_parameters.json", "w") as f:
    json.dump(
        best_params_all,
        f,
        indent=2,
        default=str
    )

print("結果を保存しました:", save_dir)

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

save_dir = "./RF_Data/RF_status_DEP"

# ==========================================
# 1. 各foldの評価指標
# ==========================================
fold_metrics_df = pd.read_csv(
    f"{save_dir}/metrics_each_fold.csv"
)

display(fold_metrics_df.round(4))

# ==========================================
# 2. 5-foldの平均・標準偏差
# ==========================================
metrics_df = pd.read_csv(
    f"{save_dir}/metrics.csv"
)

display(metrics_df.round(4))

# ==========================================
# 3. ROC曲線（各foldのOOF予測から作成）
# ==========================================
oof_df = pd.read_csv(
    f"{save_dir}/oof_predictions.csv",
    index_col="SampleID"
)

y_true = oof_df["TrueClass"].to_numpy()
y_prob = oof_df["PredictedProbability"].to_numpy()
y_pred = oof_df["PredictedClass"].to_numpy()

fpr, tpr, thresholds = roc_curve(y_true, y_prob)
auc = roc_auc_score(y_true, y_prob)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f"OOF ROC (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

# ==========================================
# 4. 混同行列
# ==========================================
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[0, 1]
)
disp.plot()
plt.title("Confusion Matrix (OOF)")
plt.show()

# ==========================================
# 5. 偽陽性・偽陰性のサンプルを確認
# ==========================================
fp_df = oof_df[
    (oof_df["TrueClass"] == 0) &
    (oof_df["PredictedClass"] == 1)
]

fn_df = oof_df[
    (oof_df["TrueClass"] == 1) &
    (oof_df["PredictedClass"] == 0)
]

print("偽陽性（False Positive）:", len(fp_df), "samples")
display(fp_df)

print("偽陰性（False Negative）:", len(fn_df), "samples")
display(fn_df)

# 必要ならCSVとして個別に保存
fp_df.to_csv(f"{save_dir}/false_positive_samples.csv")
fn_df.to_csv(f"{save_dir}/false_negative_samples.csv")

# ==========================================
# 6. 特徴量重要度
# ==========================================
importance_df = pd.read_csv(
    f"{save_dir}/feature_importance.csv"
)

display(importance_df.head(20))

top_n = 20
top_features = importance_df.head(top_n).sort_values(
    "Importance", ascending=True
)

plt.figure(figsize=(8, 7))
plt.barh(
    top_features["Protein"],
    top_features["Importance"],
    xerr=top_features["SD_Importance"]
)
plt.xlabel("Mean Feature Importance")
plt.ylabel("Protein")
plt.title(f"Top {top_n} Feature Importances")
plt.tight_layout()
plt.show()

# ==========================================
# 7. 各foldの最適パラメータ
# ==========================================
with open(
    f"{save_dir}/best_parameters.json",
    "r"
) as f:
    best_params_all = json.load(f)

best_params_df = pd.DataFrame([
    {
        "Fold": item["Fold"],
        **item["Best_Params"]
    }
    for item in best_params_all
])

display(best_params_df)

### 120人の方で検証

In [ ]:
X = pro_data_dis.loc[:, pro_data_dis.columns.isin(DEPlist.index)]

y = pro_sample_dis['groupset2']
print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
val_probs = []
val_aucs = []

fig, ax = plt.subplots()

for i, model in enumerate(modelsets_HER_DEP):

    y_val = model.predict_proba(X)[:, 1]
    val_probs.append(y_val)

    auc_val = roc_auc_score(y, y_val)
    val_aucs.append(auc_val) 

    print(f"Fold {i+1} Validation AUC: {auc_val:.4f}")

    RocCurveDisplay.from_predictions(
        y,
        y_val,
        name=f"Fold {i+1} (AUC = {auc_val:.3f})",
        ax=ax
    )

# 5モデルの予測確率を平均
val_probs = np.array(val_probs)
y_val_mean = val_probs.mean(axis=0)

# Ensemble AUC
auc_mean = roc_auc_score(y, y_val_mean)

# 平均予測のROC
RocCurveDisplay.from_predictions(
    y,
    y_val_mean,
    name=f"Mean prediction (AUC = {auc_mean:.3f})",
    ax=ax
)
# 5モデルのAUCの平均と標準偏差
auc_val_mean = np.mean(val_aucs)
auc_val_sd = np.std(val_aucs, ddof=1)

print(f"Validation AUC: {auc_val_mean:.4f} ± {auc_val_sd:.4f}")

ax.plot([0, 1], [0, 1], linestyle="--")

ax.set_title("Validation ROC curves")
plt.show()


print(f"Ensemble Validation AUC: {auc_mean:.4f}")

# 2段階RandomForest

In [ ]:
print(pro_sample["CaCo_Status"].value_counts(dropna=False))
print(pro_sample["progress_group"].value_counts(dropna=False))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix
)


In [ ]:

# ==========================================
# 1. データ準備
# ==========================================
sample_ids = pro_data.index.intersection(pro_sample.index)

X = pro_data.loc[sample_ids].copy()
meta = pro_sample.loc[sample_ids].copy()

# progress_groupを文字列に統一
pg = meta["progress_group"].astype(str).str.strip().str.lower()

stage_label = pd.Series(index=meta.index, dtype="object")

# Control
stage_label.loc[meta["CaCo_Status"] == 0] = "Control"
stage_label.loc[pg.isin(["0", "0.0", "control"])] = "Control"

# Early
stage_label.loc[
    pg.isin(["1", "1.0", "early", "early-stage", "early_stage"])
] = "Early"

# Advanced
stage_label.loc[
    pg.isin(["2", "2.0", "advanced", "advanced-stage", "advanced_stage"])
] = "Advanced"

# 有効なラベルがあるサンプルのみ使用
valid = stage_label.notna()
X = X.loc[valid]
meta = meta.loc[valid]
stage_label = stage_label.loc[valid]

y_stage = stage_label.map({
    "Control": 0,
    "Early": 1,
    "Advanced": 2
}).astype(int)

print("全サンプル数:", len(X))
print(stage_label.value_counts())

# ==========================================
# 2. 第1段階の保存結果を読み込む（再学習なし）
# ==========================================
stage1_dir = "./RF_Data/RF_status_DEP"

oof_stage1 = pd.read_csv(
    f"{stage1_dir}/oof_predictions.csv",
    index_col="SampleID"
)

metrics_stage1 = pd.read_csv(
    f"{stage1_dir}/metrics.csv"
)

print("第1段階の評価指標:")
display(metrics_stage1)

# ==========================================
# 3. 第2段階：Early vs Advanced
# ==========================================
predicted_cancer_ids = oof_stage1.index[
    oof_stage1["PredictedClass"] == 1
]

# 実際の進行度がEarlyまたはAdvancedであることも条件にする
stage2_mask = (
    X.index.isin(predicted_cancer_ids)
    & y_stage.isin([1, 2])
)

X_stage2 = X.loc[stage2_mask].copy()
y_stage2 = y_stage.loc[stage2_mask].copy()

print("第2段階のサンプル数（0と予測されたものを除いている）:", len(X_stage2))
print("Early / Advanced:")
print(y_stage2.value_counts().sort_index())


In [ ]:

# ==========================================
# 1. データ準備
# ==========================================

X = pro_data.loc[:, pro_data.columns.isin(DEPlist.index)]

meta = pro_sample.loc[sample_ids].copy()

# progress_groupを文字列に統一
pg = meta["progress_group"].astype(str).str.strip().str.lower()

stage_label = pd.Series(index=meta.index, dtype="object")

# Control
stage_label.loc[meta["CaCo_Status"] == 0] = "Control"
stage_label.loc[pg.isin(["0"])] = "Control"
# Early
stage_label.loc[pg.isin(["1"])] = "Early"

# Advanced
stage_label.loc[pg.isin(["2"])] = "Advanced"


y_stage = stage_label.map({
    "Control": 0,
    "Early": 1,
    "Advanced": 2
}).astype(int)

print("全サンプル数:", len(X))
print(stage_label.value_counts())

# ==========================================
# 2. 第1段階の保存結果を読み込む（再学習なし）
# ==========================================
stage1_dir = "./RF_Data/RF_status_DEP"

oof_stage1 = pd.read_csv(
    f"{stage1_dir}/oof_predictions.csv",
    index_col="SampleID"
)

metrics_stage1 = pd.read_csv(
    f"{stage1_dir}/metrics.csv"
)

print("第1段階の評価指標:")
display(metrics_stage1)

# ==========================================
# 3. 第2段階：Early vs Advanced
# ==========================================
predicted_cancer_ids = oof_stage1.index[
    oof_stage1["PredictedClass"] == 1
]

# 実際の進行度がEarlyまたはAdvancedであることも条件にする
stage2_mask = (
    X.index.isin(predicted_cancer_ids)
    & y_stage.isin([1, 2])
)

X_stage2 = X.loc[stage2_mask].copy()
y_stage2 = y_stage.loc[stage2_mask].copy()

print("第2段階のサンプル数（0と予測されたものを除いている）:", len(X_stage2))
print("Early / Advanced:")
print(y_stage2.value_counts().sort_index())

# 保存先
save_dir = "./RF_Data/RF_progress_DEP"
os.makedirs(save_dir, exist_ok=True)


# ==========================================
# 4. 交差検証の設定
# ==========================================
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)

param_grid = {
    "n_estimators": [50, 100, 300, 500],
    "max_depth": [None, 3, 5, 10, 15],
    "min_samples_leaf": [1, 2, 4],
    "max_features": [None, "sqrt"],
    "class_weight": ["balanced"]
}

fold_metrics = []
best_params_all = []
feature_importances = []

y_prob_all = np.full(len(y_stage2), np.nan)
y_true_all = np.full(len(y_stage2), np.nan)

fig, ax = plt.subplots(figsize=(7, 7))

# ==========================================
# 5. 5-fold交差検証
# ==========================================
for i, (train_index, test_index) in enumerate(
    skf.split(X_stage2, y_stage2)
):

    X_train = X_stage2.iloc[train_index]
    X_test = X_stage2.iloc[test_index]

    y_train = y_stage2.iloc[train_index]
    y_test = y_stage2.iloc[test_index]

    model = RandomForestClassifier(
        criterion="gini",
        random_state=123,
        n_jobs=1
    )

    cv_gen = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=123
    )

    modelcv = GridSearchCV(
        model,
        param_grid=param_grid,
        scoring="balanced_accuracy",
        n_jobs=-1,
        refit=True,
        cv=cv_gen,
        verbose=0,
        pre_dispatch="2*n_jobs",
        error_score=np.nan,
        return_train_score=False
    )

    start = time.time()
    modelcv.fit(X_train, y_train)
    elapsed = time.time() - start

    print(f"\nFold {i+1}")
    print(f"GridSearchCV時間: {elapsed / 60:.2f} 分")
    print("Best score:", modelcv.best_score_)
    print("Best parameters:", modelcv.best_params_)

    # 各foldの最適パラメータを保存
    best_params_all.append({
        "Fold": i + 1,
        "Best_Params": modelcv.best_params_
    })

    # 必要なら各foldのGridSearch結果も保存
    cv_results_df = pd.DataFrame(
        modelcv.cv_results_
    ).sort_values("mean_test_score", ascending=False)

    cv_results_df.to_csv(
        f"{save_dir}/gridsearch_fold{i+1}.csv",
        index=False
    )

    model = modelcv.best_estimator_

    # 予測確率：Advanced（ラベル2）である確率
    advanced_col = list(model.classes_).index(2)
    y_prob = model.predict_proba(X_test)[:, advanced_col]

    # 予測クラス
    y_pred = model.predict(X_test)

    # OOF予測を保存
    y_prob_all[test_index] = y_prob
    y_true_all[test_index] = y_test.to_numpy()

    # 特徴量重要度
    feature_importances.append(model.feature_importances_)

    # AUC（Advancedを陽性クラスとする）
    score = roc_auc_score(
        (y_test == 2).astype(int),
        y_prob
    )

    # 混同行列：Early=1、Advanced=2
    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred,
        labels=[1, 2]
    ).ravel()

    fold_metrics.append({
        "Fold": i + 1,
        "AUC": score,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
        "Macro F1": f1_score(
            y_test, y_pred,
            labels=[1, 2],
            average="macro",
            zero_division=0
        ),
        "Sensitivity": tp / (tp + fn) if tp + fn else np.nan,
        "Specificity": tn / (tn + fp) if tn + fp else np.nan,
        "False Positive Rate": fp / (fp + tn) if fp + tn else np.nan,
        "False Negative Rate": fn / (fn + tp) if fn + tp else np.nan,
        "PPV": tp / (tp + fp) if tp + fp else np.nan,
        "NPV": tn / (tn + fn) if tn + fn else np.nan
    })

    print("AUC:", score)

    RocCurveDisplay.from_predictions(
        (y_test == 2).astype(int),
        y_prob,
        name=f"Fold {i+1}",
        ax=ax
    )

print("\n5-fold平均AUC:",
      f"{np.mean([m['AUC'] for m in fold_metrics]):.4f} ± "
      f"{np.std([m['AUC'] for m in fold_metrics], ddof=1):.4f}")

plt.title("Early vs Advanced ROC")
plt.show()

# ==========================================
# 6. OOF予測を保存
# ==========================================
y_true_all = y_true_all.astype(int)
y_pred_all = np.where(y_prob_all >= 0.5, 2, 1)

oof_df = pd.DataFrame({
    "TrueClass": y_true_all,
    "PredictedProbability_Advanced": y_prob_all,
    "PredictedClass": y_pred_all
}, index=X_stage2.index)

oof_df.index.name = "SampleID"

oof_df.to_csv(
    f"{save_dir}/oof_predictions.csv"
)

# ==========================================
# 7. 各foldの評価指標を保存
# ==========================================
fold_metrics_df = pd.DataFrame(fold_metrics)

display(fold_metrics_df.round(4))

fold_metrics_df.to_csv(
    f"{save_dir}/metrics_each_fold.csv",
    index=False
)

# 平均・標準偏差
metric_cols = fold_metrics_df.columns.drop("Fold")

metrics_df = pd.DataFrame({
    "Metric": metric_cols,
    "Mean": [
        fold_metrics_df[col].mean()
        for col in metric_cols
    ],
    "SD": [
        fold_metrics_df[col].std(ddof=1)
        for col in metric_cols
    ]
})

display(metrics_df.round(4))

metrics_df.to_csv(
    f"{save_dir}/metrics.csv",
    index=False
)

# ==========================================
# 8. 特徴量重要度を保存
# ==========================================
importance_df = pd.DataFrame({
    "Protein": X_stage2.columns,
    "Importance": np.mean(feature_importances, axis=0),
    "SD_Importance": np.std(
        feature_importances,
        axis=0,
        ddof=1
    )
}).sort_values("Importance", ascending=False)

importance_df.to_csv(
    f"{save_dir}/feature_importance.csv",
    index=False
)

display(importance_df.head(20))

# ==========================================
# 9. 各foldの最適パラメータを保存
# ==========================================
with open(
    f"{save_dir}/best_parameters.json",
    "w"
) as f:
    json.dump(
        best_params_all,
        f,
        indent=2,
        default=str
    )

print("結果を保存しました:", save_dir)

In [ ]:

# ==========================================
# 1. データ準備
# ==========================================

X = pro_data.loc[:, pro_data.columns.isin(DEPlist.index)]

meta = pro_sample.loc[sample_ids].copy()

# progress_groupを文字列に統一
pg = meta["progress_group"].astype(str).str.strip().str.lower()

stage_label = pd.Series(index=meta.index, dtype="object")

# Control
stage_label.loc[meta["CaCo_Status"] == 0] = "Control"
stage_label.loc[pg.isin(["0"])] = "Control"
# Early
stage_label.loc[pg.isin(["1"])] = "Early"

# Advanced
stage_label.loc[pg.isin(["2"])] = "Advanced"


y_stage = stage_label.map({
    "Control": 0,
    "Early": 1,
    "Advanced": 2
}).astype(int)

print("全サンプル数:", len(X))
print(stage_label.value_counts())

# ==========================================
# 2. 第1段階の保存結果を読み込む（再学習なし）
# ==========================================
stage1_dir = "./RF_Data/RF_status_DEP"

oof_stage1 = pd.read_csv(
    f"{stage1_dir}/oof_predictions.csv",
    index_col="SampleID"
)

metrics_stage1 = pd.read_csv(
    f"{stage1_dir}/metrics.csv"
)

print("第1段階の評価指標:")
display(metrics_stage1)

# ==========================================
# 3. 第2段階：Early vs Advanced
# ==========================================
predicted_cancer_ids = oof_stage1.index[
    oof_stage1["PredictedClass"] == 1
]

# 実際の進行度がEarlyまたはAdvancedであることも条件にする
stage2_mask = (
    X.index.isin(predicted_cancer_ids)
    & y_stage.isin([1, 2])
)

X_stage2 = X.loc[stage2_mask].copy()
y_stage2 = y_stage.loc[stage2_mask].copy()

print("第2段階のサンプル数（0と予測されたものを除いている）:", len(X_stage2))
print("Early / Advanced:")
print(y_stage2.value_counts().sort_index())

# 保存先
save_dir = "./RF_Data/RF_progress_DEP"
os.makedirs(save_dir, exist_ok=True)


# ==========================================
# 4. 交差検証の設定
# ==========================================
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)

param_grid = {
    "n_estimators": [50, 100, 300, 500],
    "max_depth": [None, 3, 5, 10, 15],
    "min_samples_leaf": [2, 4, 6],
    "max_features": [None, "sqrt"],
    "class_weight": ["balanced"]
}

fold_metrics = []
best_params_all = []
feature_importances = []

y_prob_all = np.full(len(y_stage2), np.nan)
y_true_all = np.full(len(y_stage2), np.nan)

fig, ax = plt.subplots(figsize=(7, 7))

# ==========================================
# 5. 5-fold交差検証
# ==========================================
for i, (train_index, test_index) in enumerate(
    skf.split(X_stage2, y_stage2)
):

    X_train = X_stage2.iloc[train_index]
    X_test = X_stage2.iloc[test_index]

    y_train = y_stage2.iloc[train_index]
    y_test = y_stage2.iloc[test_index]

    model = RandomForestClassifier(
        criterion="gini",
        random_state=123,
        n_jobs=1
    )

    cv_gen = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=123
    )

    modelcv = GridSearchCV(
        model,
        param_grid=param_grid,
        scoring="balanced_accuracy",
        n_jobs=-1,
        refit=True,
        cv=cv_gen,
        verbose=0,
        pre_dispatch="2*n_jobs",
        error_score=np.nan,
        return_train_score=False
    )

    start = time.time()
    modelcv.fit(X_train, y_train)
    elapsed = time.time() - start

    print(f"\nFold {i+1}")
    print(f"GridSearchCV時間: {elapsed / 60:.2f} 分")
    print("Best score:", modelcv.best_score_)
    print("Best parameters:", modelcv.best_params_)

    # 各foldの最適パラメータを保存
    best_params_all.append({
        "Fold": i + 1,
        "Best_Params": modelcv.best_params_
    })

    # 必要なら各foldのGridSearch結果も保存
    cv_results_df = pd.DataFrame(
        modelcv.cv_results_
    ).sort_values("mean_test_score", ascending=False)

    cv_results_df.to_csv(
        f"{save_dir}/gridsearch_fold{i+1}.csv",
        index=False
    )

    model = modelcv.best_estimator_

    # 予測確率：Advanced（ラベル2）である確率
    advanced_col = list(model.classes_).index(2)
    y_prob = model.predict_proba(X_test)[:, advanced_col]

    # 予測クラス
    y_pred = model.predict(X_test)

    # OOF予測を保存
    y_prob_all[test_index] = y_prob
    y_true_all[test_index] = y_test.to_numpy()

    # 特徴量重要度
    feature_importances.append(model.feature_importances_)

    # AUC（Advancedを陽性クラスとする）
    score = roc_auc_score(
        (y_test == 2).astype(int),
        y_prob
    )

    # 混同行列：Early=1、Advanced=2
    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred,
        labels=[1, 2]
    ).ravel()

    fold_metrics.append({
        "Fold": i + 1,
        "AUC": score,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
        "Macro F1": f1_score(
            y_test, y_pred,
            labels=[1, 2],
            average="macro",
            zero_division=0
        ),
        "Sensitivity": tp / (tp + fn) if tp + fn else np.nan,
        "Specificity": tn / (tn + fp) if tn + fp else np.nan,
        "False Positive Rate": fp / (fp + tn) if fp + tn else np.nan,
        "False Negative Rate": fn / (fn + tp) if fn + tp else np.nan,
        "PPV": tp / (tp + fp) if tp + fp else np.nan,
        "NPV": tn / (tn + fn) if tn + fn else np.nan
    })

    print("AUC:", score)

    RocCurveDisplay.from_predictions(
        (y_test == 2).astype(int),
        y_prob,
        name=f"Fold {i+1}",
        ax=ax
    )

print("\n5-fold平均AUC:",
      f"{np.mean([m['AUC'] for m in fold_metrics]):.4f} ± "
      f"{np.std([m['AUC'] for m in fold_metrics], ddof=1):.4f}")

plt.title("Early vs Advanced ROC")
plt.show()

# ==========================================
# 6. OOF予測を保存
# ==========================================
y_true_all = y_true_all.astype(int)
y_pred_all = np.where(y_prob_all >= 0.5, 2, 1)

oof_df = pd.DataFrame({
    "TrueClass": y_true_all,
    "PredictedProbability_Advanced": y_prob_all,
    "PredictedClass": y_pred_all
}, index=X_stage2.index)

oof_df.index.name = "SampleID"

oof_df.to_csv(
    f"{save_dir}/oof_predictions.csv"
)

# ==========================================
# 7. 各foldの評価指標を保存
# ==========================================
fold_metrics_df = pd.DataFrame(fold_metrics)

display(fold_metrics_df.round(4))

fold_metrics_df.to_csv(
    f"{save_dir}/metrics_each_fold.csv",
    index=False
)

# 平均・標準偏差
metric_cols = fold_metrics_df.columns.drop("Fold")

metrics_df = pd.DataFrame({
    "Metric": metric_cols,
    "Mean": [
        fold_metrics_df[col].mean()
        for col in metric_cols
    ],
    "SD": [
        fold_metrics_df[col].std(ddof=1)
        for col in metric_cols
    ]
})

display(metrics_df.round(4))

metrics_df.to_csv(
    f"{save_dir}/metrics.csv",
    index=False
)

# ==========================================
# 8. 特徴量重要度を保存
# ==========================================
importance_df = pd.DataFrame({
    "Protein": X_stage2.columns,
    "Importance": np.mean(feature_importances, axis=0),
    "SD_Importance": np.std(
        feature_importances,
        axis=0,
        ddof=1
    )
}).sort_values("Importance", ascending=False)

importance_df.to_csv(
    f"{save_dir}/feature_importance.csv",
    index=False
)

display(importance_df.head(20))

# ==========================================
# 9. 各foldの最適パラメータを保存
# ==========================================
with open(
    f"{save_dir}/best_parameters.json",
    "w"
) as f:
    json.dump(
        best_params_all,
        f,
        indent=2,
        default=str
    )

print("結果を保存しました:", save_dir)

# CA-19-9とIL2RAのモデル

## CA-19-9とIL2RAのデータ読み込み

In [ ]:
ca199 = pro_sample[['CA19_9']].copy()
display(ca199)

il2ra = pro_sample[['sIL2R']].copy()
display(il2ra)

In [ ]:
print((ca199['CA19_9'] <= 0).sum())
print((il2ra['sIL2R'] <= 0).sum())

In [ ]:
ca199['logvalue'] = np.log10(ca199['CA19_9'])
il2ra['logvalue'] = np.log10(il2ra['sIL2R'])


## CA-19-9単独

In [ ]:
X = ca199[["logvalue"]]
y = pro_sample["CaCo_Status"]

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
modelsets_HER_ca199 = []

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
aucs = []
fig, ax = plt.subplots()

for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)
    param_grid = {
        'n_estimators': [100, 200, 300,400, 500],
        'max_depth': [None, 20, 30,40],
        'min_samples_split': [2, 3,5, 7],
        'min_samples_leaf': [1, 2, 3,4]
    }
    model = RandomForestClassifier(
        criterion='gini',
        random_state=123,
        class_weight='balanced',
        n_jobs=-1 
    )
    
    cv_gen = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
    
    modelcv = GridSearchCV(
        model, 
        param_grid=param_grid,
        scoring='roc_auc',
        n_jobs=-1, 
        refit=True, 
        cv=cv_gen,
        verbose=0, 
        pre_dispatch='2*n_jobs', 
        error_score=np.nan, 
        return_train_score=False
    )
    
    # 実行
    modelcv.fit(X_train, y_train)
    
    print('Best score: ', modelcv.best_score_)
    print('Best hyperparameters: ', modelcv.best_params_)
    dataframe = pd.DataFrame(modelcv.cv_results_).sort_values('mean_test_score', ascending=False)
    display(dataframe)
    
    model = modelcv.best_estimator_
    modelsets_HER_ca199.append(model)

    y_prob = model.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, y_prob)
    aucs.append(score)

    print('AUC: ', score)
    
    RocCurveDisplay.from_predictions(
        y_test,
        y_prob,
        name=f"Fold {i+1}", ax=ax
    )
    
print(f"5-Fold平均AUC: {np.mean(aucs):.4f} ± {np.std(aucs)}")
plt.show()

saved_assets = {
    'model': modelcv.best_estimator_
}
name = 'mdl_ca199'
save_dir = "./Data"
path = os.path.join(save_dir, name)
joblib.dump(saved_assets, path)

### 120人の方で検証

In [ ]:
X = ca199_dis[['logvalue']]

y = pro_sample_dis['groupset2']
print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
val_probs = []
val_aucs = []

fig, ax = plt.subplots()

for i, model in enumerate(modelsets_HER_ca199):

    y_val = model.predict_proba(X)[:, 1]
    val_probs.append(y_val)

    auc_val = roc_auc_score(y, y_val)
    val_aucs.append(auc_val)

    print(f"Fold {i+1} Validation AUC: {auc_val:.4f}")

    RocCurveDisplay.from_predictions(
        y,
        y_val,
        name=f"Fold {i+1} (AUC = {auc_val:.3f})",
        ax=ax
    )

# 5モデルの予測確率を平均
val_probs = np.array(val_probs)
y_val_mean = val_probs.mean(axis=0)

# Ensemble AUC
auc_mean = roc_auc_score(y, y_val_mean)

# 平均予測のROC
RocCurveDisplay.from_predictions(
    y,
    y_val_mean,
    name=f"Mean prediction (AUC = {auc_mean:.3f})",
    ax=ax
)

# 5モデルのAUCの平均 ± SD
auc_val_mean = np.mean(val_aucs)
auc_val_sd = np.std(val_aucs, ddof=1)

print(f"Validation AUC: {auc_val_mean:.4f} ± {auc_val_sd:.4f}")

ax.plot([0, 1], [0, 1], linestyle="--")

ax.set_title("Validation ROC curves")
plt.show()

print(f"Ensemble Validation AUC: {auc_mean:.4f}")

## IL2RA単独

In [ ]:
X = il2ra[["logvalue"]]
y = pro_sample["CaCo_Status"]

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
aucs = []
fig, ax = plt.subplots()

for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [None, 20, 30,40],
        'min_samples_split': [2, 3, 4],
        'min_samples_leaf': [1, 2, 3]
    }
    
    model = RandomForestClassifier(
        criterion='gini',
        random_state=123,
        class_weight='balanced',
        n_jobs=-1 
    )
    
    cv_gen = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
    
    modelcv = GridSearchCV(
        model, 
        param_grid=param_grid,
        scoring='roc_auc',
        n_jobs=-1, 
        refit=True, 
        cv=cv_gen,
        verbose=0, 
        pre_dispatch='2*n_jobs', 
        error_score=np.nan, 
        return_train_score=False
    )
    
    # 実行
    modelcv.fit(X_train, y_train)
    
    print('Best score: ', modelcv.best_score_)
    print('Best hyperparameters: ', modelcv.best_params_)
    dataframe = pd.DataFrame(modelcv.cv_results_).sort_values('mean_test_score', ascending=False)
    display(dataframe)
    
    model = modelcv.best_estimator_
    y_prob = model.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, y_prob)
    aucs.append(score)

    print('AUC: ', score)
    
    RocCurveDisplay.from_predictions(
        y_test,
        y_prob,
        name=f"Fold {i+1}", ax=ax
    )
    
print(f"5-Fold平均AUC: {np.mean(aucs):.4f} ± {np.std(aucs)}")
plt.show()

saved_assets = {
    'model': modelcv.best_estimator_
}
name = 'mdl_il2ra'
save_dir = "./Data"
path = os.path.join(save_dir, name)
joblib.dump(saved_assets, path)

## DEPlist + CA19-9

In [ ]:
dep = pro_data.loc[:, pro_data.columns.isin(DEPlist.index)]
X = pd.concat([
    ca199[['logvalue']],
    dep
], axis=1)

y = pro_sample['CaCo_Status']
print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
modelsets_HER_DEP_ca199 = []

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
aucs = []
fig, ax = plt.subplots()

for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)
    param_grid = {
        'n_estimators': [100, 200, 300,400, 500],
        'max_depth': [None, 20, 30,40],
        'min_samples_split': [2, 3,5, 7],
        'min_samples_leaf': [1, 2, 3,4]
    }
    
    model = RandomForestClassifier(
        criterion='gini',
        random_state=123,
        class_weight='balanced',
        n_jobs=-1 
    )
    
    cv_gen = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
    
    modelcv = GridSearchCV(
        model, 
        param_grid=param_grid,
        scoring='roc_auc',
        n_jobs=-1, 
        refit=True, 
        cv=cv_gen,
        verbose=0, 
        pre_dispatch='2*n_jobs', 
        error_score=np.nan, 
        return_train_score=False
    )
    
    # 実行
    modelcv.fit(X_train, y_train)
    
    print('Best score: ', modelcv.best_score_)
    print('Best hyperparameters: ', modelcv.best_params_)
    dataframe = pd.DataFrame(modelcv.cv_results_).sort_values('mean_test_score', ascending=False)
    display(dataframe)

    # 保存

    model = modelcv.best_estimator_
    modelsets_HER_DEP_ca199.append(model)

    y_prob = model.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, y_prob)
    aucs.append(score)

    print('AUC: ', score)
    
    RocCurveDisplay.from_predictions(
        y_test,
        y_prob,
        name=f"Fold {i+1}", ax=ax
    )
    
print(f"5-Fold平均AUC: {np.mean(aucs):.4f} ± {np.std(aucs)}")
plt.show()

saved_assets = {
    'model': modelcv.best_estimator_
}
name = 'mdl_DEPlist+ca199'
save_dir = "./Data"
path = os.path.join(save_dir, name)
joblib.dump(saved_assets, path)

### 120人の方で検証

In [ ]:
display(pro_data_dis)

In [ ]:
dep = pro_data_dis.loc[:, pro_data_dis.columns.isin(DEPlist.index)]
X = pd.concat([
    ca199_dis[['logvalue']],
    dep
], axis=1)

y = pro_sample_dis['groupset2']
print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
display(X)

In [ ]:
val_probs = []
val_aucs = []

fig, ax = plt.subplots()

for i, model in enumerate(modelsets_HER_DEP_ca199):

    y_val = model.predict_proba(X)[:, 1]
    val_probs.append(y_val)

    auc_val = roc_auc_score(y, y_val)
    val_aucs.append(auc_val)

    print(f"Fold {i+1} Validation AUC: {auc_val:.4f}")

    RocCurveDisplay.from_predictions(
        y,
        y_val,
        name=f"Fold {i+1} (AUC = {auc_val:.3f})",
        ax=ax
    )

# 5モデルの予測確率を平均
val_probs = np.array(val_probs)
y_val_mean = val_probs.mean(axis=0)

# Ensemble AUC
auc_mean = roc_auc_score(y, y_val_mean)

# 平均予測のROC
RocCurveDisplay.from_predictions(
    y,
    y_val_mean,
    name=f"Mean prediction (AUC = {auc_mean:.3f})",
    ax=ax
)

# 5モデルのAUCの平均と標準偏差
auc_val_mean = np.mean(val_aucs)
auc_val_sd = np.std(val_aucs, ddof=1)

print(f"Validation AUC: {auc_val_mean:.4f} ± {auc_val_sd:.4f}")

ax.plot([0, 1], [0, 1], linestyle="--")

ax.set_title("Validation ROC curves")
plt.show()

print(f"Ensemble Validation AUC: {auc_mean:.4f}")

## CA-19-9とIL2RAの複合

In [ ]:

X = pd.concat(
    [
        ca199[["logvalue"]].rename(columns={"logvalue": "ca199_logvalue"}),
        il2ra[["logvalue"]].rename(columns={"logvalue": "il2ra_logvalue"})
    ],
    axis=1
)
y = pro_sample["CaCo_Status"]

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

### 2つの変数の「差」や「和」を新たな特徴量として加える

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
aucs = []
fig, ax = plt.subplots()

for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(class_weight='balanced', random_state=123, max_iter=1000))
    ])
    
    param_grid = {
        'classifier__C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0], # 正則化の強さ（小さいほど強力に過学習を防ぐ）
        'classifier__penalty': ['l2'] # Ridge正則化（変数を程よく残す設定）
    }
    
   
    
    cv_gen = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
    
    modelcv = GridSearchCV(
        pipe, 
        param_grid=param_grid,
        scoring='roc_auc',
        n_jobs=-1, 
        refit=True, 
        cv=cv_gen,
        verbose=0, 
        error_score=np.nan, 
        return_train_score=False
    )
    
    # 実行
    modelcv.fit(X_train, y_train)
    
    print(f'--- Fold {i+1} ---')
    print('Best score: ', modelcv.best_score_)
    print('Best hyperparameters: ', modelcv.best_params_)
    
    best_pipe = modelcv.best_estimator_
    coefficients = best_pipe.named_steps['classifier'].coef_[0]
    print(f"ca199の重み: {coefficients[0]:.4f}")
    print(f"il2raの重み: {coefficients[1]:.4f}")
    
    y_prob = best_pipe.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, y_prob)
    aucs.append(score)

    print('AUC: ', score)
    RocCurveDisplay.from_predictions(
        y_test,
        y_prob,
        name=f"Fold {i+1}", ax=ax
    )
    
print(f"5-Fold平均AUC: {np.mean(aucs):.4f} ± {np.std(aucs)}")
plt.show()



# CA-19-9の値を予想するモデル作成

## IL2RAとプロテオミクスデータを結合したものを作成

In [ ]:
display(pro_data)
display(ca199)
display(il2ra)

In [ ]:
temp = pro_data.copy()
temp["IL2RA"] = il2ra["logvalue"]
display(temp["IL2RA"])
pro_data_il2ra = temp


## LASSO回帰

### 全プロテオミクス

In [ ]:
X = pro_data_il2ra
y = ca199[["logvalue"]]

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=123)
r2_list = []

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    #標準化
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    lasso_model = LassoCV(cv=5, random_state=123, n_jobs=-1)
    lasso_model.fit(X_train, y_train)
    
    r2_list.append(r2_score(y_test, lasso_model.predict(X_test)))

    # 3. 予測と評価
    y_pred = lasso_model.predict(X_test)
    print(f"Test R2 Score: {r2_score(y_test, y_pred):.3f}")
    print(f"Test MAE: {mean_absolute_error(y_test, y_pred):.3f}")
    
    # 4. どのタンパク質がCA19-9の予測に貢献したか確認
    y_pred = lasso_model.predict(X_test)
    
    
    print("=== モデル評価結果 ===")
    print(f"Alpha: {lasso_model.alpha_:.5f}")
    print(f"R2 Score: {r2_score(y_test, y_pred):.3f}")
    print(f"MAE: {mean_absolute_error(y_test, y_pred):.3f}")

    # 残差プロットの描画
    residuals = y_test.values.flatten() - y_pred
    plt.figure(figsize=(8, 5))
    plt.scatter(y_pred, residuals, alpha=0.5, color='purple')
    plt.axhline(y=0, color='r', linestyle='--')
    plt.title("Residual Plot")
    plt.xlabel("Predicted Values")
    plt.ylabel("Residuals (Actual - Predicted)")
    plt.show()

print(f"平均R2スコア: {np.mean(r2_list):.3f}")

# # 各特徴量（タンパク質）の係数（重み）を取得
# coefs = pd.Series(lasso_model.coef_, index=X.columns)

# # 係数が 0 でない（＝LASSOによって選ばれた）タンパク質だけを抽出
# selected_features = coefs[coefs != 0].sort_values(key=abs, ascending=False)

# print(f"\n全 {len(X.columns)} 個中、{len(selected_features)} 個のタンパク質が選択されました。")

# # 上位20個をグラフで可視化
# plt.figure(figsize=(10, 6))
# selected_features.head(20).sort_values().plot(kind='barh', color='salmon')
# plt.title("Top 20 Proteins Predicting CA19-9 (LASSO Coefficients)")
# plt.xlabel("Coefficient Value (Weight)")
# plt.ylabel("Proteins")
# plt.tight_layout()
# plt.show()

# print("\n--- 選択された上位タンパク質の一覧 ---")
# print(selected_features.head(20))



### DEPlist+IL2RAのみ

In [ ]:
display(DEPlist)

In [ ]:
dep = pro_data.loc[:, pro_data.columns.isin(DEPlist.index)]

X = pd.concat([
    il2ra[['logvalue']],
    dep
], axis=1)

y = ca199[["logvalue"]]

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=123)
r2_list = []

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    #標準化
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    lasso_model = LassoCV(cv=5, random_state=123, n_jobs=-1)
    lasso_model.fit(X_train, y_train)
    
    r2_list.append(r2_score(y_test, lasso_model.predict(X_test)))

    # 3. 予測と評価
    y_pred = lasso_model.predict(X_test)
    print(f"Test R2 Score: {r2_score(y_test, y_pred):.3f}")
    print(f"Test MAE: {mean_absolute_error(y_test, y_pred):.3f}")
    
    # 4. どのタンパク質がCA19-9の予測に貢献したか確認
    y_pred = lasso_model.predict(X_test)
    
    print("=== モデル評価結果 ===")
    print(f"Alpha: {lasso_model.alpha_:.5f}")
    print(f"R2 Score: {r2_score(y_test, y_pred):.3f}")
    print(f"MAE: {mean_absolute_error(y_test, y_pred):.3f}")

        # 残差プロットの描画
    residuals = y_test.values.flatten() - y_pred
    plt.figure(figsize=(8, 5))
    plt.scatter(y_pred, residuals, alpha=0.5, color='purple')
    plt.axhline(y=0, color='r', linestyle='--')
    plt.title("Residual Plot")
    plt.xlabel("Predicted Values")
    plt.ylabel("Residuals (Actual - Predicted)")
    plt.show()

print(f"平均R2スコア: {np.mean(r2_list):.3f}")

# # 各特徴量（タンパク質）の係数（重み）を取得
# coefs = pd.Series(lasso_model.coef_, index=X.columns)

# # 係数が 0 でない（＝LASSOによって選ばれた）タンパク質だけを抽出
# selected_features = coefs[coefs != 0].sort_values(key=abs, ascending=False)

# print(f"\n全 {len(X.columns)} 個中、{len(selected_features)} 個のタンパク質が選択されました。")

# # 上位20個をグラフで可視化
# plt.figure(figsize=(10, 6))
# selected_features.head(20).sort_values().plot(kind='barh', color='salmon')
# plt.title("Top 20 Proteins Predicting CA19-9 (LASSO Coefficients)")
# plt.xlabel("Coefficient Value (Weight)")
# plt.ylabel("Proteins")
# plt.tight_layout()
# plt.show()

# print("\n--- 選択された上位タンパク質の一覧 ---")
# print(selected_features.head(20))



## ランダムフォレスト

### 全プロテオミクス

In [ ]:
X = pro_data_il2ra
y = ca199[["logvalue"]]

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=123)
r2_list = []
mae_list = []
# fig, ax = plt.subplots()

for i, (train_index, test_index) in enumerate(kf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [None, 20, 30,40],
        'min_samples_split': [2, 3, 4],
        'min_samples_leaf': [1, 2, 3]
    }
    
    model = RandomForestRegressor(
        criterion='absolute_error',
        random_state=123,
        n_jobs=-1 
    )
    
    cv_gen = KFold(n_splits=5, shuffle=True, random_state=123)
    
    modelcv = GridSearchCV(
        model, 
        param_grid=param_grid,
        scoring='r2',
        n_jobs=-1, 
        refit=True, 
        cv=cv_gen,
        verbose=0, 
        pre_dispatch='2*n_jobs', 
        error_score=np.nan, 
        return_train_score=False
    )
    
    # 実行
    model.fit(X_train, y_train.values.ravel())

    y_pred = model.predict(X_test)
    
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    
    r2_list.append(r2)
    mae_list.append(mae)
    

    print(f"Fold {i+1} - Test R2 Score: {r2:.3f} | Test MAE: {mae:.3f}")

print("\n=== 全体平均結果 ===")
print(f"平均R2スコア: {np.mean(r2_list):.3f}")
print(f"平均MAE: {np.mean(mae_list):.3f}")



### DEPlist+IL2RAのみ

In [ ]:
display(DEPlist)

In [ ]:
cols = list(DEPlist.index) + ["IL2RA"]
X = pro_data_il2ra.loc[:, pro_data.columns.isin(cols)]
y = ca199[["logvalue"]]

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=123)
r2_list = []
mae_list = []
# fig, ax = plt.subplots()

for i, (train_index, test_index) in enumerate(kf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [None, 20, 30,40],
        'min_samples_split': [2, 3, 4],
        'min_samples_leaf': [1, 2, 3]
    }
    
    model = RandomForestRegressor(
        criterion='absolute_error',
        random_state=123,
        n_jobs=-1 
    )
    
    cv_gen = KFold(n_splits=5, shuffle=True, random_state=123)
    
    modelcv = GridSearchCV(
        model, 
        param_grid=param_grid,
        scoring='r2',
        n_jobs=-1, 
        refit=True, 
        cv=cv_gen,
        verbose=0, 
        pre_dispatch='2*n_jobs', 
        error_score=np.nan, 
        return_train_score=False
    )
    
    # 実行
    model.fit(X_train, y_train.values.ravel())

    y_pred = model.predict(X_test)
    
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    
    r2_list.append(r2)
    mae_list.append(mae)
    

    print(f"Fold {i+1} - Test R2 Score: {r2:.3f} | Test MAE: {mae:.3f}")

print("\n=== 全体平均結果 ===")
print(f"平均R2スコア: {np.mean(r2_list):.3f}")
print(f"平均MAE: {np.mean(mae_list):.3f}")



## 予測されたCA-19-9の値を見る

In [ ]:
display(ca199)

### 全プロテオミクス

In [ ]:
X = pro_data
y = ca199[["logvalue"]]

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=123
)

tempI = X_test.index

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', LassoCV(cv=5, random_state=123, n_jobs=-1))
])

pipeline.fit(X_train, y_train.values.ravel())


y_pred = pipeline.predict(X_test)

comparison_df = pd.DataFrame({
    'Actual_CA19_9': np.array(y_test).flatten(),
    'Predicted_CA19_9': np.array(y_pred).flatten()
}, index=tempI)
comparison_df['Abs_Diff'] = np.abs(comparison_df['Actual_CA19_9'] - comparison_df['Predicted_CA19_9'])
comparison_df['CancerStatus'] = pro_sample.loc[tempI, 'CaCo_Status']
comparison_df = comparison_df.sort_values(by='Abs_Diff', ascending=False)


# 上位50サンプルを表示
print("--- 各サンプルの実際のCA19-9 vs 予測されたCA19-9 ---")
print(comparison_df.head(50))

plt.figure(figsize=(6, 6))

# Status（0と1）でグループ分けしてプロット
colors = {0: 'teal', 1: 'orange'}
labels = {0: 'Status 0 (Healthy)', 1: 'Status 1 (Cancer)'}

for status in [0, 1]:
    subset = comparison_df[comparison_df['CancerStatus'] == status]
    plt.scatter(subset['Actual_CA19_9'], subset['Predicted_CA19_9'], 
                alpha=0.7, c=colors[status], label=labels[status])

# 完璧に予測できた場合の補助線（既存の処理）
max_val = max(comparison_df['Actual_CA19_9'].max(), comparison_df['Predicted_CA19_9'].max())
min_val = min(comparison_df['Actual_CA19_9'].min(), comparison_df['Predicted_CA19_9'].min())
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', label='Perfect Prediction')

plt.title("ALL Proteome : Actual vs Predicted CA19-9 by Status")
plt.xlabel("Actual CA19-9")
plt.ylabel("Predicted CA19-9")
plt.legend()
plt.grid(True)
plt.show()

# 1. パイプラインからLassoの係数を取り出す
coef = pipeline.named_steps['lasso'].coef_
feature_names = X.columns

# 2. データフレームにまとめて、影響度順にソート
importance_df = pd.DataFrame({
    'Protein': feature_names,
    'Coefficient': coef
}).assign(Abs_Coef = lambda x: x['Coefficient'].abs()) \
  .sort_values(by='Abs_Coef', ascending=False)

# 3. 係数が0でない（選ばれた）タンパク質のみ表示
selected_proteins = importance_df[importance_df['Coefficient'] != 0]
print(len(selected_proteins))
display(selected_proteins)

In [ ]:
from scipy.stats import pearsonr, spearmanr

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=123
)

tempI = X_test.index

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', LassoCV(cv=5, random_state=123, n_jobs=-1))
])

pipeline.fit(X_train, y_train.values.ravel())


y_pred = pipeline.predict(X_test)

comparison_df = pd.DataFrame({
    'Actual_CA19_9': np.array(y_test).flatten(),
    'Predicted_CA19_9': np.array(y_pred).flatten()
}, index=tempI)
comparison_df['Abs_Diff'] = np.abs(comparison_df['Actual_CA19_9'] - comparison_df['Predicted_CA19_9'])
comparison_df['progress'] = pro_sample.loc[tempI, 'progress']
comparison_df = comparison_df.sort_values(by='Abs_Diff', ascending=False)

actual = np.asarray(y_test).ravel()
predicted = np.asarray(y_pred).ravel()

residual = predicted - actual

print("R2:", r2_score(actual, predicted))
print("RMSE:", np.sqrt(mean_squared_error(actual, predicted)))
print("MAE:", mean_absolute_error(actual, predicted))
print("Pearson r:", pearsonr(actual, predicted).statistic)
print("Spearman rho:", spearmanr(actual, predicted).statistic)

# 上位50サンプルを表示
print("--- 各サンプルの実際のCA19-9 vs 予測されたCA19-9 ---")
print(comparison_df.head(50))

plt.figure(figsize=(6, 6))

# Status（0と1）でグループ分けしてプロット
colors = {
    0:   '#009E73',  # Healthy：青緑
    400: '#FDE725',  # 黄色
    410: '#FCA636',  # オレンジ
    420: '#F66B19',  # 橙赤
    430: '#D73027',  # 赤
    440: '#7F0000'   # 濃い赤
}
labels = {0: 'Status 0 (Healthy)', 400: 'Status 1 (Cancer)',
          410: 'Status 2 (Cancer)',420: 'Status 3 (Cancer)',
          430: 'Status 4 (Cancer)',440: 'Status 5 (Cancer)'}

for status in [0 ,400,410,420,430,440]:
    subset = comparison_df[comparison_df['progress'] == status]
    plt.scatter(subset['Actual_CA19_9'], subset['Predicted_CA19_9'], 
                alpha=0.7, c=colors[status], label=labels[status])

# 完璧に予測できた場合の補助線（既存の処理）
max_val = max(comparison_df['Actual_CA19_9'].max(), comparison_df['Predicted_CA19_9'].max())
min_val = min(comparison_df['Actual_CA19_9'].min(), comparison_df['Predicted_CA19_9'].min())
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', label='Perfect Prediction')

plt.title("ALL Proteome : Actual vs Predicted CA19-9 by Status")
plt.xlabel("Actual CA19-9")
plt.ylabel("Predicted CA19-9")
plt.legend()
plt.grid(True)
plt.show()

# 1. パイプラインからLassoの係数を取り出す
coef = pipeline.named_steps['lasso'].coef_
feature_names = X.columns

# 2. データフレームにまとめて、影響度順にソート
importance_df = pd.DataFrame({
    'Protein': feature_names,
    'Coefficient': coef
}).assign(Abs_Coef = lambda x: x['Coefficient'].abs()) \
  .sort_values(by='Abs_Coef', ascending=False)

# 3. 係数が0でない（選ばれた）タンパク質のみ表示
selected_proteins = importance_df[importance_df['Coefficient'] != 0]
print(len(selected_proteins))
display(selected_proteins)

In [ ]:
# selected proteinの詳細を見るためエクスポート
selected_proteins.to_csv("./Data/lasso_selectedProteins.csv", index=False)

In [ ]:


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=123
)
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

# 1. パイプラインの定義 (標準化 + LassoCV)
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', LassoCV(cv=5, random_state=123, n_jobs=-1))
])

# 2. 交差検証の実行 (R^2スコアを算出)
# yは.values.ravel()で1次元配列に変換します
scores = cross_val_score(pipeline, X, y.values.ravel(), cv=5, scoring='r2')

print(f"各分割のR^2スコア: {scores}")
print(f"平均R^2: {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")


### DEPlistのみ

In [ ]:
display(il2ra)

In [ ]:
cols = list(DEPlist.index)
temp = pro_data.loc[:, pro_data.columns.isin(cols)].copy()

X = pd.concat(
    [temp, il2ra["logvalue"].rename("IL2RA_logvalue")],
    axis=1
)
y = ca199[["logvalue"]]

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
from scipy.stats import pearsonr, spearmanr

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=123
)

tempI = X_test.index

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', LassoCV(cv=5, random_state=123, n_jobs=-1))
])

pipeline.fit(X_train, y_train.values.ravel())


y_pred = pipeline.predict(X_test)

comparison_df = pd.DataFrame({
    'Actual_CA19_9': np.array(y_test).flatten(),
    'Predicted_CA19_9': np.array(y_pred).flatten()
}, index=tempI)
comparison_df['Abs_Diff'] = np.abs(comparison_df['Actual_CA19_9'] - comparison_df['Predicted_CA19_9'])
comparison_df['progress'] = pro_sample.loc[tempI, 'progress']
comparison_df = comparison_df.sort_values(by='Abs_Diff', ascending=False)

actual = np.asarray(y_test).ravel()
predicted = np.asarray(y_pred).ravel()

residual = predicted - actual

print("R2:", r2_score(actual, predicted))
print("RMSE:", np.sqrt(mean_squared_error(actual, predicted)))
print("MAE:", mean_absolute_error(actual, predicted))
print("Pearson r:", pearsonr(actual, predicted).statistic)
print("Spearman rho:", spearmanr(actual, predicted).statistic)

# 上位50サンプルを表示
print("--- 各サンプルの実際のCA19-9 vs 予測されたCA19-9 ---")
print(comparison_df.head(50))

plt.figure(figsize=(6, 6))

# Status（0と1）でグループ分けしてプロット
colors = {
    0:   '#009E73',  # Healthy：青緑
    400: '#FDE725',  # 黄色
    410: '#FCA636',  # オレンジ
    420: '#F66B19',  # 橙赤
    430: '#D73027',  # 赤
    440: '#7F0000'   # 濃い赤
}
labels = {0: 'Status 0 (Healthy)', 400: 'Status 1 (Cancer)',
          410: 'Status 2 (Cancer)',420: 'Status 3 (Cancer)',
          430: 'Status 4 (Cancer)',440: 'Status 5 (Cancer)'}

for status in [0 ,400,410,420,430,440]:
    subset = comparison_df[comparison_df['progress'] == status]
    plt.scatter(subset['Actual_CA19_9'], subset['Predicted_CA19_9'], 
                alpha=0.7, c=colors[status], label=labels[status])

# 完璧に予測できた場合の補助線（既存の処理）
max_val = max(comparison_df['Actual_CA19_9'].max(), comparison_df['Predicted_CA19_9'].max())
min_val = min(comparison_df['Actual_CA19_9'].min(), comparison_df['Predicted_CA19_9'].min())
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', label='Perfect Prediction')

plt.title("DEPlist + IL2RA : Actual vs Predicted CA19-9 by Status")
plt.xlabel("Actual CA19-9")
plt.ylabel("Predicted CA19-9")
plt.legend()
plt.grid(True)
plt.show()

# 1. パイプラインからLassoの係数を取り出す
coef = pipeline.named_steps['lasso'].coef_
feature_names = X.columns

# 2. データフレームにまとめて、影響度順にソート
importance_df = pd.DataFrame({
    'Protein': feature_names,
    'Coefficient': coef
}).assign(Abs_Coef = lambda x: x['Coefficient'].abs()) \
  .sort_values(by='Abs_Coef', ascending=False)

# 3. 係数が0でない（選ばれた）タンパク質のみ表示
selected_proteins = importance_df[importance_df['Coefficient'] != 0]
print(len(selected_proteins))
display(selected_proteins)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=123
)

tempI = X_test.index

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', LassoCV(cv=5, random_state=123, n_jobs=-1))
])

pipeline.fit(X_train, y_train.values.ravel())


y_pred = pipeline.predict(X_test)

comparison_df = pd.DataFrame({
    'Actual_CA19_9': np.array(y_test).flatten(),
    'Predicted_CA19_9': np.array(y_pred).flatten()
}, index=tempI)
comparison_df['Abs_Diff'] = np.abs(comparison_df['Actual_CA19_9'] - comparison_df['Predicted_CA19_9'])
comparison_df['CancerStatus'] = pro_sample.loc[tempI, 'CaCo_Status']
comparison_df = comparison_df.sort_values(by='Abs_Diff', ascending=False)


# 上位50サンプルを表示
print("--- 各サンプルの実際のCA19-9 vs 予測されたCA19-9 ---")
print(comparison_df.head(50))

plt.figure(figsize=(6, 6))

# Status（0と1）でグループ分けしてプロット
colors = {0: 'teal', 1: 'orange'}
labels = {0: 'Status 0 (Healthy)', 1: 'Status 1 (Cancer)'}

for status in [0, 1]:
    subset = comparison_df[comparison_df['CancerStatus'] == status]
    plt.scatter(subset['Actual_CA19_9'], subset['Predicted_CA19_9'], 
                alpha=0.7, c=colors[status], label=labels[status])

# 完璧に予測できた場合の補助線（既存の処理）
max_val = max(comparison_df['Actual_CA19_9'].max(), comparison_df['Predicted_CA19_9'].max())
min_val = min(comparison_df['Actual_CA19_9'].min(), comparison_df['Predicted_CA19_9'].min())
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', label='Perfect Prediction')

plt.title("DEPlist+IL2RA : Actual vs Predicted CA19-9 by Status")
plt.xlabel("Actual CA19-9")
plt.ylabel("Predicted CA19-9")
plt.legend()
plt.grid(True)
plt.show()

# 1. パイプラインからLassoの係数を取り出す
coef = pipeline.named_steps['lasso'].coef_
feature_names = X.columns

# 2. データフレームにまとめて、影響度順にソート
importance_df = pd.DataFrame({
    'Protein': feature_names,
    'Coefficient': coef
}).assign(Abs_Coef = lambda x: x['Coefficient'].abs()) \
  .sort_values(by='Abs_Coef', ascending=False)

# 3. 係数が0でない（選ばれた）タンパク質のみ表示
selected_proteins = importance_df[importance_df['Coefficient'] != 0]
print(len(selected_proteins))
display(selected_proteins)

# SHAP value for each models

In [ ]:
%pip install --force-reinstall --no-cache-dir "shap<0.46"

In [ ]:
import numpy as np
print(np.__version__)

In [ ]:
import shap


shap.initjs()

# フォント設定
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Liberation Sans", "DejaVu Sans"]


def calculate_cv_shap(data, sample, models, data_name):

    # DEPlistのタンパク質だけを抽出
    X = data.loc[:, data.columns.isin(DEPlist.index)]
    y = sample["Cancer"]

    # モデルと特徴量の順番を確認
    print(f"\n===== {data_name} =====")
    print("X shape:", X.shape)
    print("Number of models:", len(models))

    all_model_shap_values = []

    print("Calculating SHAP values...")

    # 5モデルすべてについてSHAPを計算
    for i, model in enumerate(models):

        explainer = shap.TreeExplainer(model)
        shap_values = explainer(X)

        # クラス1（癌）のSHAP value
        if len(shap_values.shape) == 3:
            shap_values_class1 = shap_values[:, :, 1]
        else:
            shap_values_class1 = shap_values

        all_model_shap_values.append(shap_values_class1)

        print(f"Fold {i+1} completed")

    # --------------------------------
    # 5モデルのSHAP valueを平均
    # --------------------------------

    mean_shap_values = all_model_shap_values[0]

    combined_values = np.zeros_like(mean_shap_values.values)

    for sv in all_model_shap_values:
        combined_values += sv.values

    mean_shap_values.values = (
        combined_values / len(models)
    )

    # --------------------------------
    # Mean Absolute SHAP Value
    # --------------------------------

    shap_v = np.abs(
        mean_shap_values.values
    ).mean(axis=0)

    feature_names = X.columns.tolist()

    shap_importance_df = pd.DataFrame({
        "Feature": feature_names,
        "SHAP_Importance": shap_v
    })

    sorted_shap_importance_df = (
        shap_importance_df
        .sort_values(
            by="SHAP_Importance",
            ascending=False
        )
        .reset_index(drop=True)
    )

    print("\n--- SHAP Based Feature Importance ---")
    display(sorted_shap_importance_df.head(15))

    # --------------------------------
    # SHAP Summary Plot
    # --------------------------------

    plt.figure(figsize=(7, 5))

    shap.summary_plot(
        mean_shap_values,
        X,
        max_display=15,
        show=False
    )

    plt.title(
        f"SHAP Summary Plot ({data_name})",
        fontsize=13,
        pad=15
    )

    plt.tight_layout()

    plt.savefig(
        f"shap_summary_plot_cv5_{data_name}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    return sorted_shap_importance_df, mean_shap_values

In [ ]:
shap_importance_main, shap_values_main = calculate_cv_shap(
    data=pro_data,
    sample=pro_sample,
    models=modelsets_HER_DEP,
    data_name="pro_data"
)

# CA19-9とメタデータの項目の関連

In [ ]:
display(pro_sample)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))

sns.histplot(
    data=ca199,
    x="logvalue",
    bins=30,
    kde=True
)

plt.xlabel("log10(CA19-9)")
plt.ylabel("Frequency")
plt.title("Distribution of log-transformed CA19-9")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))

sns.histplot(
    data=ca199,
    x="CA19_9",
    bins=30,
    kde=True
)

plt.xlabel("(CA19-9)")
plt.ylabel("Frequency")
plt.title("Distribution of log-transformed CA19-9")
plt.tight_layout()
plt.show()

In [ ]:
ca199.describe()